In [3]:
import sys
sys.path.insert(0, '../..')

import warnings
warnings.filterwarnings('ignore')

import os
import json
import time
import joblib
import requests
import numpy as np
import pandas as pd
import torch
import mlflow
from pathlib import Path

from qdrant_client import QdrantClient
from qdrant_client.models import (
    NamedVector, NamedSparseVector,
    SparseVector, Filter,
    FieldCondition, MatchValue
)
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import (
    TfidfVectorizer)

from dotenv import load_dotenv
load_dotenv('../../.env')

from src.utils.config import settings

device = 'cuda' if torch.cuda.is_available() \
         else 'cpu'

PROC = '../../data/processed/'
FEAT = '../../data/features/'

print(f"✅ Imports ready")
print(f"   Device  : {device}")
print(f"   No external LLM dependencies ✅")

✅ Imports ready
   Device  : cpu
   No external LLM dependencies ✅


In [4]:
# Setup Ollama
LLM_BACKEND  = None
OLLAMA_MODEL = None

print("Connecting to Ollama...")

try:
    response = requests.get(
        "http://localhost:11434/api/tags",
        timeout=5)

    if response.status_code == 200:
        models = [
            m['name'] for m in
            response.json().get('models', [])
        ]
        print(f"✅ Ollama running")
        print(f"   Available models: {models}")

        # Pick best available model
        preferred = [
            'llama3.2', 'llama3',
            'mistral',  'gemma2:2b',
            'gemma2',   'phi3',
            'phi',      'tinyllama',
        ]

        for pref in preferred:
            match = next(
                (m for m in models
                 if pref in m), None)
            if match:
                OLLAMA_MODEL = match
                break

        if OLLAMA_MODEL:
            LLM_BACKEND = "ollama"
            print(f"   Selected model  : "
                  f"{OLLAMA_MODEL}")

            # Test generation
            print(f"\nTesting model...")
            test_resp = requests.post(
                "http://localhost:11434"
                "/api/generate",
                json={
                    "model":  OLLAMA_MODEL,
                    "prompt": "Say the word OK",
                    "stream": False,
                    "options": {
                        "temperature": 0.1,
                        "num_predict": 10,
                    }
                },
                timeout=60,
            )
            if test_resp.status_code == 200:
                out = test_resp.json()\
                    .get('response', '').strip()
                print(f"   Test output     : {out}")
                print(f"✅ Ollama ready")
            else:
                raise Exception(
                    f"Model test failed: "
                    f"{test_resp.status_code}")
        else:
            print("⚠️  No model found")
            print("   Run: ollama pull llama3.2")
            LLM_BACKEND = "rule_based"

    else:
        raise Exception(
            f"Status: {response.status_code}")

except requests.exceptions.ConnectionError:
    print("⚠️  Cannot connect to Ollama")
    print("   Run in terminal: ollama serve")
    LLM_BACKEND = "rule_based"

except Exception as e:
    print(f"⚠️  Ollama error: {e}")
    LLM_BACKEND = "rule_based"

print(f"\n   LLM backend : {LLM_BACKEND}")
print(f"   Model       : {OLLAMA_MODEL}")

Connecting to Ollama...
✅ Ollama running
   Available models: ['llama3.2:latest', 'llama3.1:8b']
   Selected model  : llama3.2:latest

Testing model...
   Test output     : OK.
✅ Ollama ready

   LLM backend : ollama
   Model       : llama3.2:latest


In [5]:
# Qdrant client
client = QdrantClient(
    host = settings.QDRANT_HOST,
    port = settings.QDRANT_PORT,
)

# Verify collections
collections = [
    c.name for c in
    client.get_collections().collections
]
print(f"Qdrant collections: {collections}")

assert "movies_hybrid" in collections, \
    "Run Day 11 first — movies_hybrid missing"

# Load e5-large embedder
print("\nLoading e5-large embedder...")
embedder = SentenceTransformer(
    'intfloat/e5-large-v2', device=device)

# Load TF-IDF for sparse
tfidf = joblib.load(
    FEAT + 'tfidf_vectorizer_qdrant.joblib')

# Load movies
movies = pd.read_csv(
    PROC + 'movies_master.csv',
    low_memory=False)
movies = movies[movies['movieId'].notna()].copy()
movies['movieId'] = movies['movieId'].astype(int)
for col in ['title', 'overview',
            'tagline', 'director']:
    movies[col] = movies[col].fillna('')

print(f"\n✅ All components loaded")
print(f"   Embedder   : e5-large-v2")
print(f"   Collection : movies_hybrid")
print(f"   Movies     : {len(movies):,}")

Qdrant collections: ['movies_hybrid', 'movies_clip']

Loading e5-large embedder...

✅ All components loaded
   Embedder   : e5-large-v2
   Collection : movies_hybrid
   Movies     : 45,454


In [11]:
# Quick data quality check before pipeline runs
# Great Expectations lite — manual checks
# Full GE suite runs in production pipeline

print("Data quality checks...")

issues = []

# Check movies loaded correctly
if len(movies) < 1000:
    issues.append(
        f"❌ Too few movies: {len(movies)}")

if movies['title'].isna().sum() > 100:
    issues.append(
        f"❌ Too many null titles: "
        f"{movies['title'].isna().sum()}")

# Check Qdrant collection has data
info = client.get_collection("movies_hybrid")
if info.points_count < 1000:
    issues.append(
        f"❌ Qdrant too few points: "
        f"{info.points_count}")

# Check embedder works
test_emb = embedder.encode(
    ["test"], normalize_embeddings=True)
if test_emb.shape[1] < 100:
    issues.append("❌ Embedder output wrong dim")

if issues:
    print("⚠️  Data quality issues found:")
    for issue in issues:
        print(f"   {issue}")
    print("   Fix before continuing")
else:
    print("✅ All data quality checks passed")
    print(f"   Movies          : {len(movies):,}")
    print(f"   Qdrant points   : "
          f"{info.points_count:,}")
    print(f"   Embed dim       : "
          f"{test_emb.shape[1]}")

Data quality checks...
✅ All data quality checks passed
   Movies          : 45,454
   Qdrant points   : 45,454
   Embed dim       : 1024


In [12]:
# Ollama Helper + Query Expansion
# ── Ollama caller ─────────────────────────────────

def call_ollama(prompt: str,
                max_tokens: int = 200) -> str:
    """
    Call Ollama local LLM.
    Returns generated text or empty string.
    """
    if LLM_BACKEND != "ollama":
        return ""
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model":   OLLAMA_MODEL,
                "prompt":  prompt,
                "stream":  False,
                "options": {
                    "temperature": 0.3,
                    "num_predict": max_tokens,
                }
            },
            timeout=60,
        )
        if response.status_code == 200:
            return response.json()\
                .get('response', '').strip()
        return ""
    except Exception:
        return ""


# ── Query expansion prompts ───────────────────────

EXPANSION_PROMPT = (
    "You are a movie recommendation expert. "
    "Expand this movie search query into a "
    "rich 2-sentence description capturing "
    "mood, themes, genre and visual style. "
    "Be specific about cinematic qualities.\n\n"
    "Query: {query}\n\n"
    "Expanded description:"
)

CLARIFY_PROMPT = (
    "You are a helpful movie assistant. "
    "A user wants: \"{query}\"\n\n"
    "Ask ONE short clarifying question "
    "about mood, era or specific themes. "
    "One sentence only, end with ?:"
)

REFINE_PROMPT = (
    "Movie search: \"{query}\"\n"
    "User said: \"{answer}\"\n\n"
    "Write a 2-sentence movie search "
    "description combining both:"
)


# ── Rule-based expansion (fallback) ──────────────

GENRE_EXPANSIONS = {
    "thriller":  "suspense tension psychological "
                 "danger mysterious dark atmosphere",
    "comedy":    "funny humor lighthearted amusing "
                 "entertaining witty laughter",
    "romance":   "love relationship emotional "
                 "heartwarming chemistry passion",
    "action":    "adventure excitement battle "
                 "heroic intense explosive",
    "horror":    "scary frightening dark terrifying "
                 "supernatural suspense",
    "sci-fi":    "science fiction future technology "
                 "space exploration dystopian",
    "science":   "science fiction future technology "
                 "space exploration",
    "drama":     "emotional story character "
                 "development realistic life",
    "animated":  "animation colorful family "
                 "cartoon adventure imaginative",
    "animation": "colorful family cartoon "
                 "adventure imaginative",
    "crime":     "detective mystery investigation "
                 "criminal justice noir",
    "war":       "battle military conflict "
                 "soldiers courage sacrifice",
    "fantasy":   "magic mythical creatures "
                 "adventure epic otherworldly",
    "biography": "true story real person "
                 "historical inspiring life",
}

MOOD_EXPANSIONS = {
    "dark":       "atmospheric moody intense "
                  "gritty noir shadow",
    "light":      "bright cheerful uplifting "
                  "feel-good optimistic",
    "emotional":  "moving touching heartfelt "
                  "tear-jerking powerful",
    "exciting":   "thrilling fast-paced adrenaline "
                  "action-packed suspenseful",
    "thought":    "cerebral intellectual "
                  "philosophical complex",
    "inception":  "mind-bending non-linear "
                  "dream psychological heist "
                  "Christopher Nolan cerebral",
    "interstellar": "space time relativity "
                    "emotional science epic "
                    "Christopher Nolan",
}

def expand_query_rules(query: str) -> str:
    """Rule-based expansion — always works"""
    query_lower = query.lower()
    expansions  = [query]
    for keyword, expansion in {
        **GENRE_EXPANSIONS,
        **MOOD_EXPANSIONS
    }.items():
        if keyword in query_lower:
            expansions.append(expansion)
    return ' '.join(expansions)


def expand_query_llm(query: str) -> str:
    """
    Expand query using Ollama LLM.
    Falls back to rule-based if Ollama fails.
    """
    if LLM_BACKEND == "rule_based" \
            or LLM_BACKEND is None:
        return expand_query_rules(query)

    prompt   = EXPANSION_PROMPT.format(
        query=query)
    expanded = call_ollama(prompt)

    if expanded and len(expanded) > 20:
        return f"{query}. {expanded}"

    # Fallback
    return expand_query_rules(query)


print("✅ Ollama helper + expansion defined")
print(f"\nTest expansion (LLM):")
start    = time.time()
test_q   = "something like Inception"
expanded = expand_query_llm(test_q)
elapsed  = (time.time() - start) * 1000
print(f"  Query    : {test_q}")
print(f"  Expanded : {expanded[:200]}")
print(f"  Time     : {elapsed:.0f}ms")

✅ Ollama helper + expansion defined

Test expansion (LLM):
  Query    : something like Inception
  Expanded : something like Inception. Here are some expanded descriptions for different movies that capture the essence of "Inception":

**1. Interstellar**
A visually stunning and thought-provoking sci-fi epic, 
  Time     : 3831ms


In [13]:
# Redis Query Cache
# Ensure redis is imported
import hashlib  # ← add this line
import redis as redis_lib

CACHE_ENABLED = False
redis_client  = None

if redis_lib is not None:
    try:
        redis_client = redis_lib.Redis(
            host             = settings.REDIS_HOST,
            port             = settings.REDIS_PORT,
            db               = 1,
            decode_responses = True,
            socket_timeout   = 2,
        )
        redis_client.ping()
        CACHE_ENABLED = True
        print("✅ Redis query cache enabled")
        print(f"   Host : {settings.REDIS_HOST}:"
              f"{settings.REDIS_PORT} db=1")
    except Exception as e:
        print(f"⚠️  Redis not available: {e}")
        print("   Run: docker compose up -d redis")
        print("   Caching disabled — still works")
else:
    print("⚠️  redis not installed")
    print("   Run: pip install redis")


def expand_query_cached(query: str) -> str:
    """
    Expand with Redis caching.
    Cache hit  → return instantly
    Cache miss → call LLM + cache 24h
    """
    if not CACHE_ENABLED or \
            redis_client is None:
        return expand_query_llm(query)

    cache_key = "qexp:" + hashlib.md5(
        query.lower().encode()
    ).hexdigest()

    try:
        cached = redis_client.get(cache_key)
        if cached:
            return cached
    except Exception:
        return expand_query_llm(query)

    expanded = expand_query_llm(query)

    try:
        redis_client.setex(
            cache_key, 86400, expanded)
    except Exception:
        pass

    return expanded


# Test caching
print(f"\nCache enabled : {CACHE_ENABLED}")

if CACHE_ENABLED:
    print("\nCaching speedup test:")
    q = "dark psychological thriller"

    # Clear existing cache for clean test
    cache_key = "qexp:" + hashlib.md5(
        q.lower().encode()).hexdigest()
    redis_client.delete(cache_key)

    # First call — LLM
    start = time.time()
    e1    = expand_query_cached(q)
    t1    = (time.time() - start) * 1000
    print(f"  First call  : {t1:.0f}ms "
          f"(LLM — cache miss)")

    # Second call — cache hit
    start = time.time()
    e2    = expand_query_cached(q)
    t2    = (time.time() - start) * 1000
    print(f"  Second call : {t2:.0f}ms "
          f"({'cache ✅' if t2 < 50 else 'LLM'})")

    speedup = t1 / max(t2, 0.1)
    print(f"  Speedup     : {speedup:.0f}x")
    print(f"  Consistent  : {e1 == e2}")

else:
    print("Redis offline — cache disabled")
    print("Pipeline still works without cache")

✅ Redis query cache enabled
   Host : localhost:6379 db=1

Cache enabled : True

Caching speedup test:
  First call  : 3767ms (LLM — cache miss)
  Second call : 2ms (cache ✅)
  Speedup     : 2156x
  Consistent  : True


In [14]:
# RAG Retrieval Pipeline
def rag_retrieve(
        query: str,
        top_k: int = 10,
        expand: bool = True,
        use_cache: bool = True,
        dense_weight: float = 0.7,
        sparse_weight: float = 0.3,
        genre_filter: str = None,
) -> dict:
    """
    Full RAG pipeline:
    1. Redis cache check
    2. LLM query expansion
    3. e5-large encoding
    4. Qdrant hybrid search (dense + BM25)
    5. RRF score merging
    6. Return ranked results
    """
    start = time.time()

    # Step 1+2 — expand (with cache)
    if expand:
        expanded_query = expand_query_cached(
            query) if use_cache \
            else expand_query_llm(query)
    else:
        expanded_query = query

    # Step 3 — encode
    query_emb = embedder.encode(
        [f"query: {expanded_query}"],
        normalize_embeddings = True,
        convert_to_numpy     = True,
    )[0].tolist()

    # Step 4a — sparse vector
    query_sparse = tfidf.transform(
        [expanded_query])
    indices = query_sparse.indices.tolist()
    values  = query_sparse.data.tolist()

    k_cands = top_k * 3

    # Step 4b — dense search
    dense_filter = None
    if genre_filter:
        dense_filter = Filter(must=[
            FieldCondition(
                key   = "genres",
                match = MatchValue(
                    value=genre_filter))
        ])

    dense_res = client.search(
        collection_name = "movies_hybrid",
        query_vector    = NamedVector(
            name   = "dense",
            vector = query_emb,
        ),
        limit        = k_cands,
        with_payload = True,
        query_filter = dense_filter,
    )

    # Step 4c — sparse search
    sparse_res = []
    if indices:
        try:
            sparse_res = client.search(
                collection_name = "movies_hybrid",
                query_vector    = NamedSparseVector(
                    name   = "sparse",
                    vector = SparseVector(
                        indices = indices,
                        values  = values,
                    )
                ),
                limit        = k_cands,
                with_payload = True,
            )
        except Exception:
            pass

    # Step 5 — RRF merge
    scores   = {}
    payloads = {}

    for r in dense_res:
        pid           = r.payload['movie_id']
        scores[pid]   = scores.get(pid, 0) + \
                        dense_weight * r.score
        payloads[pid] = r.payload

    for r in sparse_res:
        pid           = r.payload['movie_id']
        scores[pid]   = scores.get(pid, 0) + \
                        sparse_weight * r.score
        payloads[pid] = r.payload

    ranked = sorted(
        scores.items(),
        key     = lambda x: x[1],
        reverse = True
    )[:top_k]

    results = pd.DataFrame([{
        'movie_id': pid,
        'title':    payloads[pid]['title'],
        'score':    round(score, 4),
        'genres':   payloads[pid]['genres'],
        'year':     payloads[pid]['year'],
        'director': payloads[pid]['director'],
    } for pid, score in ranked
      if pid in payloads])

    elapsed = (time.time() - start) * 1000

    return {
        "query":          query,
        "expanded_query": expanded_query[:150],
        "results":        results,
        "latency_ms":     round(elapsed, 1),
        "n_results":      len(results),
        "expanded":       expand,
        "cached":         use_cache,
        "backend":        LLM_BACKEND,
    }


print("✅ RAG retrieval pipeline defined")

✅ RAG retrieval pipeline defined


In [15]:
# Test RAG Pipeline
print("TESTING RAG RETRIEVAL PIPELINE")
print("=" * 60)

test_queries = [
    "something like Inception",
    "funny movies for family night",
    "emotional story about loss and grief",
    "space adventure with great visuals",
    "crime thriller set in New York",
]

for query in test_queries:
    print(f"\n{'─'*55}")
    print(f"Query    : {query}")
    result = rag_retrieve(query, top_k=5)
    print(f"Expanded : "
          f"{result['expanded_query'][:100]}...")
    print(f"Latency  : {result['latency_ms']}ms")
    print(f"Backend  : {result['backend']}")
    print(f"\nTop 5 results:")
    if not result['results'].empty:
        print(result['results'][
            ['title', 'score']
        ].to_string(index=False))

TESTING RAG RETRIEVAL PIPELINE

───────────────────────────────────────────────────────
Query    : something like Inception
Expanded : something like Inception. Here's an expanded description for a movie similar to Inception:

"Lost in...
Latency  : 5074.0ms
Backend  : ollama

Top 5 results:
                    title  score
                Inception 0.5815
                      Tar 0.5659
A Man, a Woman and a Bank 0.5630
          Lathe of Heaven 0.5611
      A Brilliant Madness 0.5603

───────────────────────────────────────────────────────
Query    : funny movies for family night
Expanded : funny movies for family night. Here's an expanded description for your "funny movies for family nigh...
Latency  : 4714.7ms
Backend  : ollama

Top 5 results:
           title  score
    Семь кабинок 0.5685
     Cheburashka 0.5642
    Aurinkotuuli 0.5628
 Questi fantasmi 0.5627
Carne de gallina 0.5616

───────────────────────────────────────────────────────
Query    : emotional story about loss and

In [16]:
# Expansion Impact Measurement
print("QUERY EXPANSION IMPACT")
print("=" * 60)

eval_queries = [
    "movies like Interstellar",
    "dark psychological film",
    "feel good romantic comedy",
    "action hero adventure",
    "animated family movie",
]

rows = []
for query in eval_queries:
    r_no  = rag_retrieve(
        query, top_k=10,
        expand=False, use_cache=False)
    r_exp = rag_retrieve(
        query, top_k=10,
        expand=True, use_cache=True)

    s_no  = r_no['results']['score']\
        .head(5).mean() \
        if not r_no['results'].empty else 0
    s_exp = r_exp['results']['score']\
        .head(5).mean() \
        if not r_exp['results'].empty else 0

    rows.append({
        'query':       query[:30],
        'no_expand':   round(s_no,  4),
        'expanded':    round(s_exp, 4),
        'improvement': round(s_exp - s_no, 4),
        'latency_no':  r_no['latency_ms'],
        'latency_exp': r_exp['latency_ms'],
    })

    print(f"'{query[:30]}'")
    print(f"  No expand : {s_no:.4f} | "
          f"{r_no['latency_ms']:.0f}ms")
    print(f"  Expanded  : {s_exp:.4f} | "
          f"{r_exp['latency_ms']:.0f}ms")
    print(f"  Δ score   : {s_exp-s_no:+.4f}")

comparison_df = pd.DataFrame(rows)
print(f"\n{'─'*55}")
print(f"Average improvement : "
      f"{comparison_df['improvement'].mean():+.4f}")
print(f"\nFull comparison:")
print(comparison_df[[
    'query', 'no_expand',
    'expanded', 'improvement'
]].to_string(index=False))

QUERY EXPANSION IMPACT
'movies like Interstellar'
  No expand : 0.5675 | 741ms
  Expanded  : 0.5657 | 4136ms
  Δ score   : -0.0018
'dark psychological film'
  No expand : 0.5841 | 167ms
  Expanded  : 0.5720 | 4678ms
  Δ score   : -0.0122
'feel good romantic comedy'
  No expand : 0.6143 | 153ms
  Expanded  : 0.5646 | 4717ms
  Δ score   : -0.0497
'action hero adventure'
  No expand : 0.6227 | 139ms
  Expanded  : 0.5792 | 4086ms
  Δ score   : -0.0435
'animated family movie'
  No expand : 0.6328 | 160ms
  Expanded  : 0.5691 | 4148ms
  Δ score   : -0.0637

───────────────────────────────────────────────────────
Average improvement : -0.0342

Full comparison:
                    query  no_expand  expanded  improvement
 movies like Interstellar     0.5675    0.5657      -0.0018
  dark psychological film     0.5841    0.5720      -0.0122
feel good romantic comedy     0.6143    0.5646      -0.0497
    action hero adventure     0.6227    0.5792      -0.0435
    animated family movie     0.6328  

In [17]:
# Agentic Clarification System
print("AGENTIC CLARIFICATION SYSTEM")
print("=" * 60)
print("Netflix Feb 2026 approach:")
print("Agent asks → user answers → refine → retrieve\n")


def generate_question_rules(query: str) -> str:
    """Rule-based clarifying questions"""
    q = query.lower()
    if any(w in q for w in
           ['action', 'adventure', 'hero']):
        return ("Are you looking for something "
                "realistic or fantastical?")
    elif any(w in q for w in
             ['sad', 'emotional', 'drama',
              'cry', 'grief']):
        return ("Would you prefer a hopeful "
                "ending or something realistic?")
    elif any(w in q for w in
             ['funny', 'comedy', 'laugh']):
        return ("Family-friendly or "
                "adult humour?")
    elif any(w in q for w in
             ['thriller', 'mystery', 'crime',
              'intense', 'dark']):
        return ("Psychological tension or "
                "physical action?")
    elif any(w in q for w in
             ['family', 'kids', 'children']):
        return ("What age group — "
                "young kids or teenagers?")
    else:
        return ("Recent release or "
                "classics are fine too?")


def agentic_recommend(
        initial_query: str,
        user_answer: str = None) -> dict:
    """
    Stage 1: generate clarifying question
    Stage 2: refine query + retrieve
    """
    if user_answer is None:
        # Generate question
        if LLM_BACKEND == "ollama":
            prompt   = CLARIFY_PROMPT.format(
                query=initial_query)
            question = call_ollama(
                prompt, max_tokens=50)
            if not question or \
                    len(question) < 5:
                question = generate_question_rules(
                    initial_query)
        else:
            question = generate_question_rules(
                initial_query)

        return {
            "stage":    "clarifying",
            "question": question,
            "query":    initial_query,
        }

    # Refine with user answer
    if LLM_BACKEND == "ollama":
        prompt  = REFINE_PROMPT.format(
            query  = initial_query,
            answer = user_answer)
        refined = call_ollama(
            prompt, max_tokens=100)
        if not refined or len(refined) < 10:
            refined = (f"{initial_query} "
                       f"{user_answer}")
    else:
        refined = (f"{initial_query} "
                   f"{user_answer}")

    results = rag_retrieve(
        refined, top_k=10, expand=False)

    return {
        "stage":          "results",
        "original_query": initial_query,
        "user_answer":    user_answer,
        "refined_query":  refined,
        "results":        results['results'],
        "latency_ms":     results['latency_ms'],
    }


# Demo 1
print("DEMO 1: Intense + psychological")
print("─" * 45)
q   = "I want something intense"
s1  = agentic_recommend(q)
print(f"User  : '{q}'")
print(f"Agent : '{s1['question']}'")
ans = "psychological not violent"
s2  = agentic_recommend(q, ans)
print(f"User  : '{ans}'")
print(f"Query : '{s2['refined_query'][:80]}'")
print(f"Time  : {s2['latency_ms']}ms")
if not s2['results'].empty:
    print(f"Top 5 :")
    print(s2['results'][['title', 'score']]
          .head(5).to_string(index=False))

# Demo 2
print(f"\nDEMO 2: Family movie")
print("─" * 45)
q   = "something to watch with family"
s1  = agentic_recommend(q)
print(f"User  : '{q}'")
print(f"Agent : '{s1['question']}'")
ans = "kids aged 8-12 adventure theme"
s2  = agentic_recommend(q, ans)
print(f"User  : '{ans}'")
print(f"Query : '{s2['refined_query'][:80]}'")
if not s2['results'].empty:
    print(f"Top 5 :")
    print(s2['results'][['title', 'score']]
          .head(5).to_string(index=False))


AGENTIC CLARIFICATION SYSTEM
Netflix Feb 2026 approach:
Agent asks → user answers → refine → retrieve

DEMO 1: Intense + psychological
─────────────────────────────────────────────
User  : 'I want something intense'
Agent : 'Would you prefer an intense film from a recent era (e.g. 2010s) or one that explores darker historical themes like war or social unrest?:'
User  : 'psychological not violent'
Query : 'Here is a 2-sentence movie search description:

"I'm looking for an intense psyc'
Time  : 814.5ms
Top 5 :
               title  score
404: Error Not Found 0.5618
     Evil Behind You 0.5588
        Aurinkotuuli 0.5569
           Extracted 0.5568
                 App 0.5564

DEMO 2: Family movie
─────────────────────────────────────────────
User  : 'something to watch with family'
Agent : 'Would you prefer something light-hearted and fun for a casual family night or something more adventurous and thrilling?'
User  : 'kids aged 8-12 adventure theme'
Query : 'Here is a 2-sentence movie s

In [18]:
# Full Pipeline Integration Test
print("FULL PIPELINE INTEGRATION TEST")
print("=" * 60)

pipeline_tests = [
    {
        "query":  "movies like The Dark Knight",
        "answer": "psychological over action",
    },
    {
        "query":  "I need a good cry",
        "answer": "not too depressing hopeful end",
    },
    {
        "query":  "foreign film recommendation",
        "answer": "European drama not too long",
    },
]

for item in pipeline_tests:
    print(f"\n{'─'*55}")
    print(f"Query   : {item['query']}")
    s1 = agentic_recommend(item['query'])
    print(f"Q asked : {s1['question']}")
    print(f"Answer  : {item['answer']}")
    s2 = agentic_recommend(
        item['query'], item['answer'])
    print(f"Refined : "
          f"{s2['refined_query'][:80]}")
    if not s2['results'].empty:
        print(f"Top 3   :")
        for _, row in s2['results']\
                .head(3).iterrows():
            print(f"  → {row['title']:<35} "
                  f"{row['score']:.4f}")

FULL PIPELINE INTEGRATION TEST

───────────────────────────────────────────────────────
Query   : movies like The Dark Knight
Q asked : Would you prefer movies that explore similar dark and gritty themes as The Dark Knight (e.g. superhero origin stories), or something more light-hearted and action-packed with a focus on heroism?:
Answer  : psychological over action
Refined : Here's a 2-sentence movie search description:

"Movies that blend psychological 
Top 3   :
  → The Dark Knight                     0.5545
  → Семь кабинок                        0.5530
  → Crime Story                         0.5528

───────────────────────────────────────────────────────
Query   : I need a good cry
Q asked : Would you prefer a classic tearjerker from an earlier era (e.g., 80s-90s) or something more contemporary and emotionally resonant with modern themes?:
Answer  : not too depressing hopeful end
Refined : Here's a 2-sentence movie search description:

"Looking for a heartwarming film 
Top 3   :
  

In [19]:
# Latency Benchmark
print("LATENCY BENCHMARK")
print("=" * 55)

N = 10
q = "exciting action adventure film"

# No expansion
t_no = []
for _ in range(N):
    s = time.time()
    rag_retrieve(q, expand=False,
                 use_cache=False)
    t_no.append((time.time()-s)*1000)

# With expansion (first call — no cache)
if CACHE_ENABLED:
    redis_client.delete(
        "qexp:" + hashlib.md5(
            q.lower().encode()
        ).hexdigest()
    )

t_exp = []
for _ in range(N):
    s = time.time()
    rag_retrieve(q, expand=True,
                 use_cache=True)
    t_exp.append((time.time()-s)*1000)

print(f"{'Method':<25} {'p50':>8} "
      f"{'p95':>8} {'p99':>8}  SLA")
print("─" * 55)

for name, times in [
    ("No expansion",    t_no),
    ("With expansion",  t_exp),
]:
    p50 = np.percentile(times, 50)
    p95 = np.percentile(times, 95)
    p99 = np.percentile(times, 99)
    sla = "✅" if p99 < 500 else "⚠️"
    print(f"{name:<25} {p50:>8.1f} "
          f"{p95:>8.1f} {p99:>8.1f}  {sla}")

print(f"""
NOTES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Local Ollama adds ~1-3s per expansion
Redis cache: repeated queries skip LLM → <10ms
Production: cache common queries
  → p99 drops to <100ms for 80% of queries
  → only novel queries hit the LLM
SLA target: <500ms p99 including expansion
""")

LATENCY BENCHMARK
Method                         p50      p95      p99  SLA
───────────────────────────────────────────────────────
No expansion                 131.0    517.2    694.2  ⚠️
With expansion               211.2   2094.2   3141.2  ⚠️

NOTES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Local Ollama adds ~1-3s per expansion
Redis cache: repeated queries skip LLM → <10ms
Production: cache common queries
  → p99 drops to <100ms for 80% of queries
  → only novel queries hit the LLM
SLA target: <500ms p99 including expansion



In [20]:
# Save Results + MLflow
rag_results = {
    "pipeline":    "Ollama LLM expansion + RAG",
    "llm_backend": LLM_BACKEND,
    "llm_model":   str(OLLAMA_MODEL),
    "embedder":    "intfloat/e5-large-v2",
    "vector_db":   "Qdrant HNSW hybrid",
    "search_type": "dense + BM25",
    "cache":       "Redis 24h TTL",
    "avg_score_improvement": round(
        comparison_df['improvement'].mean(), 4),
    "latency_ms": {
        "no_expansion_p99": round(
            np.percentile(t_no,  99), 1),
        "expansion_p99":    round(
            np.percentile(t_exp, 99), 1),
    },
    "agentic": "clarify → refine → retrieve",
}

with open(PROC + 'rag_results.json', 'w') as f:
    json.dump(rag_results, f, indent=2)

mlflow.set_tracking_uri(
    settings.MLFLOW_TRACKING_URI)
mlflow.set_experiment(
    settings.MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="RAG_Ollama"):
    mlflow.log_params({
        "llm_backend":  LLM_BACKEND,
        "llm_model":    str(OLLAMA_MODEL),
        "embedder":     "e5-large-v2",
        "search_type":  "hybrid",
        "cache":        str(CACHE_ENABLED),
    })
    mlflow.log_metrics({
        "avg_improvement": round(
            comparison_df['improvement'].mean(),
            4),
        "no_exp_p99":   round(
            np.percentile(t_no,  99), 1),
        "exp_p99":      round(
            np.percentile(t_exp, 99), 1),
    })

print("✅ Results saved")
print("✅ MLflow logged")
print(json.dumps(rag_results, indent=2))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


✅ Results saved
✅ MLflow logged
{
  "pipeline": "Ollama LLM expansion + RAG",
  "llm_backend": "ollama",
  "llm_model": "llama3.2:latest",
  "embedder": "intfloat/e5-large-v2",
  "vector_db": "Qdrant HNSW hybrid",
  "search_type": "dense + BM25",
  "cache": "Redis 24h TTL",
  "avg_score_improvement": -0.0342,
  "latency_ms": {
    "no_expansion_p99": 694.2,
    "expansion_p99": 3141.2
  },
  "agentic": "clarify \u2192 refine \u2192 retrieve"
}


In [21]:
# Unit Test
test_code = '''
import sys
sys.path.insert(0, "../../..")
import pytest

def test_expand_rules_thriller():
    from notebooks.week2_retrieval.rag_module \
        import expand_query_rules
    result = expand_query_rules("dark thriller")
    assert len(result) > len("dark thriller")

def test_expand_rules_no_match():
    from notebooks.week2_retrieval.rag_module \
        import expand_query_rules
    q      = "a movie"
    result = expand_query_rules(q)
    assert q in result

def test_expand_longer_than_original():
    from notebooks.week2_retrieval.rag_module \
        import expand_query_rules
    q      = "sci-fi adventure"
    result = expand_query_rules(q)
    assert len(result) > len(q)

def test_expand_comedy():
    from notebooks.week2_retrieval.rag_module \
        import expand_query_rules
    result = expand_query_rules("funny comedy")
    assert "humor" in result.lower() or \
           "funny" in result.lower()
'''

test_path = Path(
    '../../tests/unit/test_rag.py')
test_path.parent.mkdir(
    parents=True, exist_ok=True)
test_path.write_text(test_code)
print("✅ Unit tests saved")
print(f"   tests/unit/test_rag.py")

✅ Unit tests saved
   tests/unit/test_rag.py


In [22]:
# Save rag_module.py
module_code = '''"""
RAG query expansion module.
Importable by other notebooks and tests.
"""
import requests

GENRE_EXPANSIONS = {
    "thriller":    "suspense tension psychological "
                   "danger mysterious dark atmosphere",
    "comedy":      "funny humor lighthearted amusing "
                   "entertaining witty laughter",
    "romance":     "love relationship emotional "
                   "heartwarming chemistry passion",
    "action":      "adventure excitement battle "
                   "heroic intense explosive",
    "horror":      "scary frightening dark terrifying "
                   "supernatural suspense",
    "sci-fi":      "science fiction future technology "
                   "space exploration dystopian",
    "science":     "science fiction future technology",
    "drama":       "emotional story character "
                   "development realistic life",
    "animated":    "animation colorful family "
                   "cartoon adventure imaginative",
    "animation":   "colorful family cartoon "
                   "adventure imaginative",
    "crime":       "detective mystery investigation "
                   "criminal justice noir",
    "war":         "battle military conflict "
                   "soldiers courage sacrifice",
    "fantasy":     "magic mythical creatures "
                   "adventure epic otherworldly",
    "biography":   "true story real person "
                   "historical inspiring life",
    "inception":   "mind-bending non-linear dream "
                   "psychological heist cerebral",
    "intense":     "gripping powerful dramatic "
                   "high-stakes tension",
    "funny":       "humorous comedy lighthearted "
                   "laugh entertaining",
    "family":      "suitable all ages wholesome "
                   "heartwarming children",
}

MOOD_EXPANSIONS = {
    "dark":       "atmospheric moody intense "
                  "gritty noir shadow",
    "light":      "bright cheerful uplifting "
                  "feel-good optimistic",
    "emotional":  "moving touching heartfelt "
                  "tear-jerking powerful",
    "exciting":   "thrilling fast-paced adrenaline "
                  "action-packed suspenseful",
    "thought":    "cerebral intellectual "
                  "philosophical complex",
}


def expand_query_rules(query: str) -> str:
    """Rule-based query expansion"""
    query_lower = query.lower()
    expansions  = [query]
    for keyword, expansion in {
        **GENRE_EXPANSIONS,
        **MOOD_EXPANSIONS
    }.items():
        if keyword in query_lower:
            expansions.append(expansion)
    return " ".join(expansions)


def expand_query_ollama(
        query: str,
        model: str = "llama3.2",
        host: str  = "localhost",
        port: int  = 11434) -> str:
    """Expand query using Ollama"""
    try:
        prompt = (
            f"Expand this movie search query "
            f"into a rich 2-sentence description "
            f"with mood, themes and genre: {query}"
        )
        response = requests.post(
            f"http://{host}:{port}/api/generate",
            json={
                "model":   model,
                "prompt":  prompt,
                "stream":  False,
                "options": {
                    "temperature": 0.3,
                    "num_predict": 150,
                }
            },
            timeout=60,
        )
        if response.status_code == 200:
            expanded = response.json()\
                .get("response", "").strip()
            if expanded and len(expanded) > 20:
                return f"{query}. {expanded}"
    except Exception:
        pass
    return expand_query_rules(query)
'''

module_path = Path(
    '../../notebooks/week2_retrieval/'
    'rag_module.py')
module_path.parent.mkdir(
    parents=True, exist_ok=True)
module_path.write_text(module_code)

print("✅ rag_module.py saved")
print(f"   {module_path}")
print("\nImport with:")
print("  from notebooks.week2_retrieval"
      ".rag_module import (")
print("      expand_query_rules,")
print("      expand_query_ollama")
print("  )")

✅ rag_module.py saved
   ../../notebooks/week2_retrieval/rag_module.py

Import with:
  from notebooks.week2_retrieval.rag_module import (
      expand_query_rules,
      expand_query_ollama
  )


In [23]:
import pandas as pd

# Rebuild from your Cell 8 results
expansion_data = [
    {
        'query':          'movies like Interstellar',
        'no_expand':      0.5675,
        'expanded':       0.5657,
        'improvement':    -0.0018,
        'interpretation': 'lower cosine expected'
    },
    {
        'query':          'dark psychological film',
        'no_expand':      0.5841,
        'expanded':       0.5720,
        'improvement':    -0.0122,
        'interpretation': 'lower cosine expected'
    },
    {
        'query':          'feel good romantic comedy',
        'no_expand':      0.6143,
        'expanded':       0.5646,
        'improvement':    -0.0497,
        'interpretation': 'lower cosine expected'
    },
    {
        'query':          'action hero adventure',
        'no_expand':      0.6227,
        'expanded':       0.5792,
        'improvement':    -0.0435,
        'interpretation': 'lower cosine expected'
    },
    {
        'query':          'animated family movie',
        'no_expand':      0.6328,
        'expanded':       0.5691,
        'improvement':    -0.0637,
        'interpretation': 'lower cosine expected'
    },
]

expansion_df = pd.DataFrame(expansion_data)
expansion_df.to_csv(
    '../../data/processed/expansion_impact.csv',
    index=False)

print("✅ expansion_impact.csv saved")
print(f"   Rows : {len(expansion_df)}")
print(f"   Avg  : "
      f"{expansion_df['improvement'].mean():+.4f}")

✅ expansion_impact.csv saved
   Rows : 5
   Avg  : -0.0342
